# The Lagrange points from $\nabla U_{\rm eff} = 0$

The handout's Step 6(a) says a body can sit still in the rotating frame exactly where the
effective potential has zero gradient,

$$U_{\rm eff} = U - \tfrac12 m\Omega^2 (x^2 + y^2),\qquad
U = -\frac{GmM_1}{\sqrt{(x + r_1)^2 + y^2}} - \frac{GmM_2}{\sqrt{(x - r_2)^2 + y^2}} .$$

This notebook builds $U_{\rm eff}$ with [sympy](https://docs.sympy.org/latest/index.html), takes the gradient symbolically, plots it,
and finds the five zeros numerically. Units as in the handout's "Nailing down details": $R = 1$, $G(M_1 + M_2) = 1$,
so $\Omega = 1$, $M_1 \to 1 - \mu$ at $x = -\mu$ and $M_2 \to \mu$ at $x = 1 - \mu$, everything
per unit test mass $m$.

## 1. Setup

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp
from sympy import init_printing
from scipy.optimize import brentq, fsolve
init_printing()

mu = 0.1      # M2/(M1+M2); 0.1 is the handout's figure, 3.00e-6 is the Sun and the Earth

## 2. $U_{\rm eff}$ and its gradient, symbolically

In [ ]:
x, y = sp.symbols('x y', real=True)
d1 = sp.sqrt((x + mu)**2 + y**2)          # distance to M1
d2 = sp.sqrt((x - 1 + mu)**2 + y**2)      # distance to M2
U_eff = -(1 - mu)/d1 - mu/d2 - (x**2 + y**2)/2

grad = [sp.diff(U_eff, x), sp.diff(U_eff, y)]
grad

Each component is (gravity from $M_1$) + (gravity from $M_2$) + (centrifugal), the handout's
equation (1.12) with the signs flipped because this is $+\nabla U_{\rm eff}$ ([`diff`](https://docs.sympy.org/latest/tutorials/intro-tutorial/calculus.html#derivatives) did the
derivatives). [`lambdify`](https://docs.sympy.org/latest/modules/utilities/lambdify.html) turns them into ordinary numpy functions:

In [ ]:
U_num = sp.lambdify((x, y), U_eff, 'numpy')
gx = sp.lambdify((x, y), grad[0], 'numpy')
gy = sp.lambdify((x, y), grad[1], 'numpy')

## 3. A picture of $U_{\rm eff}$

In [ ]:
X, Y = np.meshgrid(np.linspace(-1.6, 1.6, 400), np.linspace(-1.6, 1.6, 400))
Z = U_num(X, Y)
Z = np.clip(Z, -3, None)       # the potential goes to -infinity at the primaries; cut it off for the plot

plt.figure(figsize=(6, 6))
plt.contour(X, Y, Z, levels=np.linspace(-3, -1.4, 40), linewidths=0.6)
plt.plot([-mu, 1 - mu], [0, 0], 'ko')
for px, lab in [(-mu, '$M_1$'), (1 - mu, '$M_2$')]:
    plt.text(px, -0.18, lab, ha='center', bbox=dict(fc='white', ec='none', pad=1))
plt.gca().set_aspect('equal'); plt.xlabel('x / R'); plt.ylabel('y / R')
plt.title('$U_{\\rm eff}$, $\\mu$ = %g' % mu);

## 4. The three on the axis

On $y = 0$ the $y$ component of the gradient is zero by itself (every term carries a factor
of $y$), so only the $x$ component is left: one equation in $x$, the handout's (1.17). Plot it
and look for the zero crossings.

We have to be slightly clever here. We just want to see a slice of what the gradient looks like along the $x$ axis. And we're going to want to look for where that gradient is zero. If we plot it naively, we get a big 'ol spike at each of the primaries because $U$ blows up there. So, we think ahead and blank out those spikes. Procedurally, matplotlib won't plot things that are nans, so we set large values of $f$ to nan.

In [ ]:
xs = np.linspace(-1.8, 1.8, 2000)
f = gx(xs, 0.0)
f = np.where(np.abs(f) < 5, f, np.nan)    # blank out the spikes at the two primaries

plt.figure(figsize=(7, 3.5))
plt.plot(xs, f); plt.axhline(0, color='gray', lw=0.8)
plt.axvline(-mu, color='k', ls=':', lw=1, label='$M_1$ at $x = -\\mu$')         # the primaries
plt.axvline(1 - mu, color='k', ls='--', lw=1, label='$M_2$ at $x = 1 - \\mu$')
plt.legend(loc='upper left')
plt.ylim(-5, 5); plt.xlabel('x / R'); plt.ylabel(r'$\partial U_{\rm eff}/\partial x$ on $y=0$');

We see three crossings: one left of $M_1$, one between the primaries, one right of $M_2$. [`brentq`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.brentq.html)
finds a zero in an interval where the sign changes, so give it those three intervals, stopping
just short of each primary where the function blows up.

In [ ]:
eps = 1e-6
L3 = brentq(lambda s: gx(s, 0.0), -2.0,      -mu - eps)
L1 = brentq(lambda s: gx(s, 0.0), -mu + eps,  1 - mu - eps)
L2 = brentq(lambda s: gx(s, 0.0),  1 - mu + eps, 2.5)
print('L1 = %.6f   L2 = %.6f   L3 = %.6f' % (L1, L2, L3))
print('distance of L1, L2 from M2: %.4f, %.4f;  Hill radius (mu/3)^(1/3) = %.4f'
      % (1 - mu - L1, L2 - (1 - mu), (mu/3)**(1/3)))

## 5. The two off the axis

Off the axis both components of the gradient have to vanish, two equations in two unknowns.
[`fsolve`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.fsolve.html) needs a starting guess; the handout says $L_4$ and $L_5$ make equilateral triangles
with the primaries, so start near the third corners, $(\tfrac12 - \mu,\ \pm\tfrac{\sqrt3}{2})$,
and see where the solver lands.

`fsolve` wants one function that takes the unknowns as a single array `p` (here `p[0]` is $x$
and `p[1]` is $y$) and returns a list of the same length that should all be zero. So
`grad_Ueff` below just bundles the two gradient components from section 2 into that form;
section 4 needed only `gx` because $y = 0$ killed the other one.

In [ ]:
def grad_Ueff(p):
    return [gx(p[0], p[1]), gy(p[0], p[1])]

L4 = fsolve(grad_Ueff, [0.5 - mu + 0.05, 0.8])
L5 = fsolve(grad_Ueff, [0.5 - mu + 0.05, -0.8])
print('L4 = (%.6f, %.6f)   equilateral corner = (%.6f, %.6f)' % (L4[0], L4[1], 0.5 - mu, np.sqrt(3)/2))
print('L5 = (%.6f, %.6f)' % (L5[0], L5[1]))

## 6. All five on the picture

In [ ]:
plt.figure(figsize=(6, 6))
plt.contour(X, Y, Z, levels=np.linspace(-3, -1.4, 40), linewidths=0.6)
plt.plot([-mu, 1 - mu], [0, 0], 'ko')
for name, (px, py) in {'L1': (L1, 0), 'L2': (L2, 0), 'L3': (L3, 0), 'L4': L4, 'L5': L5}.items():
    plt.plot(px, py, 'o', mfc='white', mec='red')
    plt.text(px, py + 0.08, name, ha='center', color='red')
plt.gca().set_aspect('equal'); plt.xlabel('x / R'); plt.ylabel('y / R')
plt.title('the five zeros of $\\nabla U_{\\rm eff}$, $\\mu$ = %g' % mu);

## 7. Things to try

- Set `mu = 3.003e-6` (the Sun and the Earth) and rerun. $L_1$ and $L_2$ sit $0.0100\,R$ from
  the Earth, $1.50\times10^6$ km, where SOHO and JWST are. The picture is no longer useful
  (the points sit on top of the Earth at this scale), but the numbers are.
- Set `mu = 0.0122` (the Earth and the Moon) and compare $L_1$, $L_2$ with the handout's table:
  the Hill-radius formula is five percent off there.
- Change the starting guess for [`fsolve`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.fsolve.html) to something far from the equilateral corner and see
  where it lands; a root finder gives you a root, not necessarily the one you meant.